# Control y Evasion de Colisiones — MyCobot 280
### Rol 3: Ingeniero de Control — Alex
**Problemas que cubre:** P4 (Evasion de Colisiones) y P5 (Control de Trayectorias)

---
**Orden de ejecucion:**
1. Celda 1 — Conexion al robot
2. Celda 2 — Poses clave
3. Celda 3 — Botones de control
4. Celda 4 — P4: Evasion de colisiones
5. Celda 5 — P5: Funciones de movimiento
6. Celda 6 — P4: Pruebas de colision
7. Celda 7 — P5: Ciclo de agarre (5 repeticiones)
8. Celda 8 — Intercambio de cubos

In [ ]:
# ============================================================
# CELDA 1 — CONEXION AL ROBOT
# Inicializa la conexion con el MyCobot 280 via puerto USB
# Puerto: /dev/ttyUSB0 | Baud rate: 1000000
# ============================================================

from pymycobot.mycobot import MyCobot
import ipywidgets.widgets as widgets
from IPython.display import display
import datetime
import time

# Conectar al robot
mc = MyCobot('/dev/ttyUSB0', 1000000)
mc.power_on()
time.sleep(1)

# Verificar conexion
if mc.is_controller_connected():
    print("Robot conectado correctamente")
else:
    print("ERROR: No se pudo conectar al robot")

print("Angulos actuales:", mc.get_angles())
print("Coordenadas actuales:", mc.get_coords())

In [ ]:
# ============================================================
# CELDA 2 — POSES CLAVE DEL CICLO (P5)
# Angulos medidos fisicamente con el robot en el laboratorio
# Formato: [J1, J2, J3, J4, J5, J6] en grados
# ============================================================

# Velocidad de movimiento (20 = lento y seguro para pruebas)
VELOCIDAD = 20

# Pose de reposo segura — siempre comenzar aqui
pose_inicial   = [0.7,   -0.17,  -0.61,  -1.58,  -0.35, -44.29]

# Pose de observacion — robot apunta hacia el area de trabajo
pose_observar  = [-3.25, -60.02, -1.05, -22.06,   6.41, -44.38]

# Pose de agarre — extremo sobre el objeto a recoger
pose_agarrar   = [-2.1,  -81.29, -1.23,   1.05,   6.41, -44.29]

# Pose de deposito — zona donde se deja el objeto
pose_depositar = [89.12, -72.15, -1.23,  -4.21,   4.48, -44.12]

# Mover a pose inicial al cargar
mc.send_angles(pose_inicial, VELOCIDAD)
time.sleep(3)

print("Poses cargadas correctamente")
print("pose_inicial   :", pose_inicial)
print("pose_observar  :", pose_observar)
print("pose_agarrar   :", pose_agarrar)
print("pose_depositar :", pose_depositar)

In [ ]:
# ============================================================
# CELDA 3 — BOTONES DE CONTROL
# Parada de emergencia y reinicio del sistema
# IMPORTANTE: Correr esta celda antes de ejecutar cualquier ciclo
# ============================================================

# Variable global que bloquea todos los movimientos cuando es True
PARAR = False

boton_parar = widgets.Button(
    description='PARAR EMERGENCIA',
    button_style='danger',
    layout=widgets.Layout(width='250px', height='45px')
)

boton_continuar = widgets.Button(
    description='CONTINUAR',
    button_style='success',
    layout=widgets.Layout(width='250px', height='45px')
)

output_estado = widgets.Output()

def parar_clicked(b):
    global PARAR
    PARAR = True
    # Vuelve a pose segura al activar emergencia
    mc.send_angles(pose_inicial, 20)
    with output_estado:
        print("[" + datetime.datetime.now().strftime('%H:%M:%S') + "] 🛑 PARADA DE EMERGENCIA ACTIVADA")

def continuar_clicked(b):
    global PARAR
    PARAR = False
    with output_estado:
        print("[" + datetime.datetime.now().strftime('%H:%M:%S') + "] Sistema listo para continuar")

boton_parar.on_click(parar_clicked)
boton_continuar.on_click(continuar_clicked)

display(widgets.HBox([boton_parar, boton_continuar]))
display(output_estado)
print("Botones de control listos")

In [ ]:
# ============================================================
# CELDA 4 — P4: EVASION DE COLISIONES
#
# Escenario 1 — Colision con la mesa:
#   Medicion real: Z=116mm el robot toca la mesa
#   Limite seguro definido: Z_MIN = 125mm
#
# Escenario 2 — Colision del robot consigo mismo:
#   Verifica que los angulos no excedan los limites fisicos
#   del MyCobot 280 segun tabla DH del fabricante
#
# Estrategia elegida: verificacion previa + altura minima
#   - Verifica angulos ANTES de ejecutar send_angles()
#   - Verifica altura Z DESPUES de cada movimiento
#   - Si detecta peligro, vuelve automaticamente a pose_inicial
# ============================================================

# Altura minima medida en laboratorio (choca en Z=116mm, margen=125mm)
Z_MIN = 125

# Limites articulares del MyCobot 280 (fuente: tabla DH del fabricante)
LIMITES_ARTICULARES = {
    'J1': (-168, 168),
    'J2': (-135,  90),
    'J3': (-150, 150),
    'J4': (-145, 145),
    'J5': (-165, 165),
    'J6': (-180, 180),
}

# Limites cartesianos del espacio de trabajo (mm)
LIMITES_COORDS = {
    'X':  (-280, 280),
    'Y':  (-280, 280),
    'Z':  (125,  419),  # Z_MIN medido = 124.7mm
    'RX': (-180, 180),
    'RY': (-180, 180),
    'RZ': (-180, 180),
}

def verificar_colision_angulos(pose):
    """
    Escenario 2: Verifica que los angulos esten dentro
    de los limites fisicos del robot antes de mover.
    Retorna True si es seguro, False si hay riesgo.
    """
    for i, angulo in enumerate(pose):
        lo, hi = LIMITES_ARTICULARES[f'J{i+1}']
        if not (lo <= angulo <= hi):
            print("❌ COLISION DETECTADA — angulo fuera de limite")
            print(f"   J{i+1}: {angulo} grados | limite: [{lo}, {hi}]")
            print("   Movimiento bloqueado")
            return False
    return True

def verificar_colision_mesa():
    """
    Escenario 1: Verifica que el extremo no baje de Z_MIN.
    Medicion real: Z=116mm toca la mesa. Limite seguro: Z=125mm.
    Retorna True si es seguro, False si hay riesgo.
    """
    coords = mc.get_coords()
    if coords is None:
        print("ADVERTENCIA: No se pudo leer coordenadas")
        return True  # No bloquear si no puede leer
    z_actual = coords[2]
    if z_actual < Z_MIN:
        print("❌ COLISION CON MESA DETECTADA")
        print(f"   Z actual: {z_actual}mm | Z minimo: {Z_MIN}mm")
        print("   Volviendo a pose segura...")
        mc.send_angles(pose_inicial, 20)
        time.sleep(3)
        return False
    return True

def verificar_colision_coords(coords):
    """
    Verifica que las coordenadas cartesianas esten
    dentro del espacio de trabajo seguro definido.
    """
    ejes = ['X', 'Y', 'Z', 'RX', 'RY', 'RZ']
    for i, eje in enumerate(ejes):
        lo, hi = LIMITES_COORDS[eje]
        valor = coords[i]
        if not (lo <= valor <= hi):
            print("❌ COLISION DETECTADA — coordenada fuera de limite")
            print(f"   Eje {eje}: {valor} | limite: [{lo}, {hi}]")
            print("   Volviendo a pose segura...")
            mc.send_angles(pose_inicial, 20)
            time.sleep(3)
            return False
    return True

print("P4 - Evasion de colisiones cargada")
print(f"  Escenario 1: Z minimo = {Z_MIN}mm (choca en Z=116mm)")
print(f"  Escenario 2: Limites articulares J1-J6 del MyCobot 280")

In [ ]:
# ============================================================
# CELDA 5 — P5: FUNCIONES DE MOVIMIENTO
# Todas las funciones integran la evasion de colisiones (P4)
# y el registro en log para el reporte de metricas
# ============================================================

# Log de sesion para registrar exito/fallo por fase
LOG = []

def goto_pose(pose, nombre="pose", velocidad=VELOCIDAD):
    """
    Mueve el robot a una pose verificando:
    1. Parada de emergencia activa
    2. Angulos dentro de limites (P4 Escenario 2)
    3. Altura Z despues de mover (P4 Escenario 1)
    4. Coordenadas dentro del espacio de trabajo
    Registra resultado en LOG.
    """
    global PARAR
    hora = datetime.datetime.now().strftime('%H:%M:%S')

    # Verificar parada de emergencia
    if PARAR:
        print("[BLOQUEADO] Parada de emergencia activa")
        return False

    # Verificar angulos antes de mover (Escenario 2)
    if not verificar_colision_angulos(pose):
        LOG.append(f"[{hora}] BLOQUEADO {nombre} — angulos invalidos")
        return False

    # Ejecutar movimiento
    print(f"Moviendo a {nombre}: {pose}")
    mc.send_angles(pose, velocidad)
    time.sleep(3)

    # Verificar altura despues de mover (Escenario 1)
    if not verificar_colision_mesa():
        LOG.append(f"[{hora}] COLISION {nombre} — Z por debajo del limite")
        return False

    # Verificar coordenadas del espacio de trabajo
    coords = mc.get_coords()
    if coords and not verificar_colision_coords(coords):
        LOG.append(f"[{hora}] COLISION {nombre} — coordenadas fuera de rango")
        return False

    print(f"Pose alcanzada: {mc.get_angles()}")
    LOG.append(f"[{hora}] OK {nombre}")
    return True

def pick():
    """Secuencia de agarre: observar -> abrir gripper -> bajar -> cerrar gripper"""
    print("--- Iniciando agarre ---")
    if not goto_pose(pose_observar, "OBSERVAR"): return False
    mc.set_gripper_value(100, 50)  # Abrir gripper
    time.sleep(1)
    if not goto_pose(pose_agarrar, "AGARRAR"): return False
    mc.set_gripper_value(0, 50)    # Cerrar gripper
    time.sleep(1)
    print("Objeto agarrado")
    return True

def place():
    """Secuencia de deposito: mover al deposito -> abrir gripper"""
    print("--- Iniciando deposito ---")
    if not goto_pose(pose_depositar, "DEPOSITAR"): return False
    mc.set_gripper_value(100, 50)  # Abrir gripper
    time.sleep(1)
    print("Objeto depositado")
    return True

def abrir_gripper():
    """Abre el gripper completamente (valor 100)"""
    mc.set_gripper_value(100, 50)
    time.sleep(1)

def cerrar_gripper():
    """Cierra el gripper completamente (valor 0)"""
    mc.set_gripper_value(0, 50)
    time.sleep(1)

print("Funciones de movimiento cargadas: goto_pose, pick, place")

In [ ]:
# ============================================================
# CELDA 6 — P4: PRUEBAS DE EVASION DE COLISIONES
# Valida que el sistema detecta y bloquea situaciones peligrosas
# ============================================================

# Pose limite medida en laboratorio (Z=124.7mm, justo antes de chocar)
pose_limite_mesa = [-3.77, -77.51, -1.75, -10.63, 6.76, -43.59]

# PRUEBA 1: Angulo fuera de limite — debe bloquearse sin mover
print("=" * 50)
print("PRUEBA 1: Angulo invalido (debe bloquearse)")
print("=" * 50)
pose_invalida = [0, -200, 0, 0, 0, 0]  # J2=-200 excede limite de -135
verificar_colision_angulos(pose_invalida)

# PRUEBA 2: Bajar hasta el limite de la mesa y detectar colision
print("\n" + "=" * 50)
print("PRUEBA 2: Limite de mesa (debe detectar colision)")
print("=" * 50)
print(f"Bajando a Z=124.7mm (choca en Z=116mm, limite={Z_MIN}mm)...")
goto_pose(pose_limite_mesa, "LIMITE_MESA")

# PRUEBA 3: Movimiento seguro normal — debe completarse sin problemas
print("\n" + "=" * 50)
print("PRUEBA 3: Movimiento seguro (debe completarse)")
print("=" * 50)
goto_pose(pose_observar, "OBSERVAR")

# Volver a pose inicial al terminar las pruebas
goto_pose(pose_inicial, "INICIAL")

In [ ]:
# ============================================================
# CELDA 7 — P5: CICLO DE AGARRE (5 repeticiones)
# Ejecuta el ciclo completo: inicial -> observar -> agarrar
# -> depositar -> inicial
# Registra tiempo, exito/fallo por fase y tasa de exito
# ============================================================

def run_cycle(n=5):
    """
    Ejecuta n ciclos consecutivos de agarre y deposito.
    Registra en LOG: tiempo de ciclo, exito/fallo por fase.
    """
    global PARAR, LOG
    PARAR = False
    LOG = []
    resultados = []

    for i in range(n):
        inicio = datetime.datetime.now()
        print("\n" + "=" * 50)
        print(f"   CICLO {i+1} de {n}")
        print("=" * 50)

        try:
            # Fase 1: Pose inicial
            if not goto_pose(pose_inicial, "INICIAL"):   break

            # Fase 2: Agarre (observar + bajar + cerrar gripper)
            if not pick(): break

            # Fase 3: Deposito (mover + abrir gripper)
            if not place(): break

            # Fase 4: Volver a pose inicial
            if not goto_pose(pose_inicial, "INICIAL"): break

            # Registrar resultado exitoso con tiempo
            fin = datetime.datetime.now()
            duracion = (fin - inicio).seconds
            msg = f"Ciclo {i+1}: EXITO — {duracion}s"
            resultados.append(msg)
            LOG.append(f"[{fin.strftime('%H:%M:%S')}] {msg}")
            print(f"Ciclo {i+1} completado en {duracion}s")

        except Exception as e:
            # Registrar fallo y volver a pose segura
            msg = f"Ciclo {i+1}: FALLO — {e}"
            resultados.append(msg)
            LOG.append(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] {msg}")
            print(f"Error en ciclo {i+1}: {e}")
            goto_pose(pose_inicial, "INICIAL")  # Emergencia

    # Tabla de resultados finales
    print("\n" + "=" * 50)
    print("   RESULTADOS FINALES")
    print("=" * 50)
    for r in resultados:
        print(r)
    exitos = sum(1 for r in resultados if "EXITO" in r)
    print(f"\nTasa de exito: {exitos}/{n} ({exitos/n*100:.0f}%)")

    # Log completo de sesion
    print("\n" + "=" * 50)
    print("   LOG DE SESION")
    print("=" * 50)
    for entrada in LOG:
        print(entrada)

# Ejecutar 5 ciclos consecutivos
run_cycle(5)

In [ ]:
# ============================================================
# CELDA 8 — INTERCAMBIO DE CUBOS
# Mueve el cubo A de su posicion original al lado opuesto
# Poses medidas fisicamente en el laboratorio
# ============================================================

# Poses del Cubo A (lado izquierdo de la mesa)
# Secuencia: encima -> antes de agarrar -> agarrar -> subir -> dejar -> despues de dejar
A_encima        = [114.6,  -45.0,  -60.38, 16.96,  9.14,  -15.73]  # Posicion encima del cubo
A_antes_agarrar = [115.13, -54.05, -60.38, 19.51,  9.14,  -15.73]  # Aproximacion al cubo
A_agarrar       = [115.13, -53.87, -60.2,  20.3,   9.05,  -15.73]  # Agarre del cubo
A_subir         = [115.13, -54.05, -60.38, 19.51,  9.14,  -15.73]  # Subir con el cubo
A_dejar         = [-87.89, -45.17, -60.73, 12.04,  9.22,  -34.54]  # Posicion de deposito
A_despues_dejar = [-87.89, -39.11, -60.73, 12.3,   9.75,  -34.27]  # Subir despues de soltar

def mover_cubo_A(repeticiones=2):
    """
    Mueve el cubo A de su posicion original al lado opuesto.
    Repite el ciclo el numero de veces indicado.
    """
    for i in range(repeticiones):
        print("\n" + "=" * 50)
        print(f"   CUBO A — CICLO {i+1} de {repeticiones}")
        print("=" * 50)

        # Abrir gripper y aproximarse al cubo
        abrir_gripper()
        if not goto_pose(A_encima,        "A_ENCIMA"):        break
        if not goto_pose(A_antes_agarrar, "A_ANTES_AGARRAR"): break
        if not goto_pose(A_agarrar,       "A_AGARRAR"):       break

        # Agarrar y subir
        cerrar_gripper()
        if not goto_pose(A_subir,         "A_SUBIR"):         break

        # Depositar en el lado opuesto
        if not goto_pose(A_dejar,         "A_DEJAR"):         break
        abrir_gripper()
        if not goto_pose(A_despues_dejar, "A_DESPUES_DEJAR"): break

        # Volver a pose inicial
        goto_pose(pose_inicial, "INICIAL")
        print(f"Ciclo {i+1} completado")

# Ejecutar 2 ciclos de intercambio del cubo A
mover_cubo_A(2)